# 🛢️ Oil Pulse Pipeline — Step-by-Step Run

Run each cell in order to execute the full pipeline.  
You'll see the output of every step inline.

**Pipeline flow:**  
`Ingest (prices + news + trump + war)` → `Spark Clean` → `Spark Features` → `Spark Aggregate` → `Train Model` → `Predict` → `War Exit Model` → `Load DuckDB` → `Dashboard`

---
## 📦 Step 0: Verify Environment

In [8]:
import sys, os
import pandas as pd
from pathlib import Path
import glob


# Fix for Windows: set HADOOP_HOME and add to PATH before Spark loads
os.environ["HADOOP_HOME"] = r"C:\hadoop"
if r"C:\hadoop\bin" not in os.environ.get("PATH", ""):
    os.environ["PATH"] = r"C:\hadoop\bin" + os.pathsep + os.environ.get("PATH", "")

print(f"Python: {sys.version}")
print(f"Working dir: {os.getcwd()}")
print(f"HADOOP_HOME: {os.environ.get('HADOOP_HOME')}")

# Quick import check
import pyspark, duckdb, yfinance, feedparser, vaderSentiment, streamlit, plotly
print(f"PySpark: {pyspark.__version__}")
print(f"DuckDB:  {duckdb.__version__}")
print("\n✅ All packages available!")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working dir: c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline
HADOOP_HOME: C:\hadoop
PySpark: 3.5.5
DuckDB:  1.5.1

✅ All packages available!


---
## 📥 Step 1: Ingest — Fetch Oil Prices
Fetches crude oil futures (CL=F) from Yahoo Finance.  
First run: last 90 days. Subsequent runs: last 2 days.

In [5]:
!python scripts/ingest/fetch_oil_prices.py

Saved 1 rows to c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\..\..\data\raw\prices\2026-04-05.csv
Output: c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\..\..\data\raw\prices\2026-04-05.csv


c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\fetch_oil_prices.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


In [ ]:
# View the raw price data

price_files = sorted(glob.glob("data/raw/prices/*.csv"))
print(f"Price files found: {len(price_files)}")
if price_files:
    df = pd.read_csv(price_files[-1])
    print(f"Latest file: {price_files[-1]} ({len(df)} rows)")
    print(f"Date range: {df['date'].min()} → {df['date'].max()}")
    df.head(10)

Price files found: 1
Latest file: data/raw/prices\2026-04-05.csv (1 rows)
Date range: 2026-04-02 → 2026-04-02


In [7]:
# Arrange and display raw price file contents in a clean format
if not price_files:
    print("No price files found.")
else:
    print(f"Total price files: {len(price_files)}")
    
    for i, file in enumerate(price_files, 1):
        print("\n" + "=" * 80)
        print(f"[{i}] File: {file}")
        
        df_file = pd.read_csv(file)
        
        # Arrange columns (if present)
        preferred_order = ["date", "open", "high", "low", "close", "volume"]
        cols = [c for c in preferred_order if c in df_file.columns] + [c for c in df_file.columns if c not in preferred_order]
        df_file = df_file[cols]
        
        # Sort by date for better readability
        if "date" in df_file.columns:
            df_file["date"] = pd.to_datetime(df_file["date"], errors="coerce")
            df_file = df_file.sort_values("date").reset_index(drop=True)
        
        print(f"Rows: {len(df_file)} | Columns: {list(df_file.columns)}")
        print("-" * 80)
        print(df_file.to_string(index=False))

Total price files: 1

[1] File: data/raw/prices\2026-04-05.csv
Rows: 1 | Columns: ['date', 'open', 'high', 'low', 'close', 'volume']
--------------------------------------------------------------------------------
      date      open       high  low      close  volume
2026-04-02 98.919998 113.970001 97.5 112.059998  514620


---
## 📰 Step 2: Ingest — Fetch RSS News
Fetches business news from BBC, CNBC, and MarketWatch RSS feeds.  
Each article is scored for sentiment using VADER.

In [5]:
!python scripts/ingest/fetch_rss_news.py

Fetched 30 articles from cnbc
Fetched 56 articles from bbc
Fetched 10 articles from marketwatch
Saved 96 articles to c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\..\..\data\raw\news\2026-04-05.csv
Output: c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\..\..\data\raw\news\2026-04-05.csv


c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\fetch_rss_news.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date": datetime.utcnow().strftime("%Y-%m-%d"),
c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ingest\fetch_rss_news.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().strftime("%Y-%m-%d")


In [ ]:
# View the raw news data
news_files = sorted(glob.glob("data/raw/news/*.csv"))
print(f"News files found: {len(news_files)}")
if news_files:
    df_news = pd.read_csv(news_files[-1])
    print(f"Latest file: {news_files[-1]} ({len(df_news)} rows)")
    print(f"Sources: {df_news['source'].value_counts().to_dict()}")
    print(f"\nSentiment stats:")
    print(df_news['sentiment_compound'].describe())
    print(f"\nSample headlines:")


News files found: 1
Latest file: data/raw/news\2026-04-05.csv (96 rows)
Sources: {'bbc': 56, 'cnbc': 30, 'marketwatch': 10}

Sentiment stats:
count    96.000000
mean     -0.083902
std       0.519134
min      -0.957100
25%      -0.499625
50%      -0.070800
75%       0.318200
max       0.865800
Name: sentiment_compound, dtype: float64

Sample headlines:
0    U.S. payrolls rose by 178,000 in March, more t...
1    The March jobs report will be released on Frid...
2    Private sector hiring totaled 62,000 in March,...
3    Why $4 a gallon gas prices won’t trigger Fed i...
4    China suppliers warn of higher prices for Amer...
5    New fees, fewer flights: Higher fuel prices pi...
6    Markets now see the Fed's next move as a poten...
7    China industrial profits surge 15% to start ye...
8    Global forecasting group sees U.S. inflation a...
9    Recession odds climb on Wall Street as economy...
Name: title, dtype: object


In [1]:
print(df_news[["title","summary"]].to_string(index=False).split("\n")[1])

NameError: name 'df_news' is not defined

---
## 🏛️ Step 2b: Ingest — Fetch Trump Statements
Fetches articles quoting Trump on oil, war, and Middle East from Google News RSS.  
Each article is tagged by topic (oil, iran, war, sanctions) and scored for sentiment.  
Captures Truth Social posts via news coverage — no Twitter/Truth API needed.

In [ ]:
!python scripts/ingest/fetch_trump_statements.py

In [ ]:
# View Trump statement data
trump_files = sorted(glob.glob("data/raw/trump/*.csv"))
print(f"Trump files found: {len(trump_files)}")
if trump_files:
    df_t = pd.read_csv(trump_files[-1])
    print(f"Latest: {trump_files[-1]} ({len(df_t)} articles)")
    print(f"Topics: {df_t['topic_tags'].value_counts().head(10).to_dict()}")
    print(f"\nSentiment stats:")
    print(df_t['sentiment_compound'].describe())
    print(f"\nSample headlines:")
    for _, row in df_t.head(5).iterrows():
        sent = row['sentiment_compound']
        print(f"  [{sent:+.2f}] {row['title'][:100]}")

---
## ⚔️ Step 2c: Ingest — Fetch War/Conflict News
Fetches Middle East war/conflict news from Google News RSS and Al Jazeera.  
Each article is classified by stance: **favor** (positive/de-escalation), **against** (escalation), or **neutral**.

In [ ]:
!python scripts/ingest/fetch_war_news.py

In [ ]:
# View war news data
war_files = sorted(glob.glob("data/raw/war_news/*.csv"))
print(f"War news files found: {len(war_files)}")
if war_files:
    df_w = pd.read_csv(war_files[-1])
    print(f"Latest: {war_files[-1]} ({len(df_w)} articles)")
    print(f"Stance breakdown: {df_w['stance'].value_counts().to_dict()}")
    print(f"\nSources: {df_w['source'].value_counts().head(10).to_dict()}")
    print(f"\nSample headlines:")
    for _, row in df_w.head(5).iterrows():
        print(f"  [{row['stance']:>7}] {row['title'][:100]}")

---
## 🧹 Step 3: Spark — Clean Raw Data
Loads all raw CSVs with explicit schemas (StructType), handles nulls, deduplicates, normalizes dates.  
Output: Parquet files in `data/processed/clean/`

In [2]:
!python scripts/transform/spark_clean.py

Cleaned: prices=1, reddit=0, news=92
93
SUCCESS: The process with PID 4404 (child process of PID 20632) has been terminated.
SUCCESS: The process with PID 20632 (child process of PID 19468) has been terminated.
SUCCESS: The process with PID 19468 (child process of PID 18704) has been terminated.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 1) / 1]

[Stage 2:>                                                          (0 + 1) / 1]

                                                                                

[Stage 9:>                                                          (0 + 1) / 1]

                                                                                


In [5]:
# Verify cleaned output
clean_dir = Path("data/processed/clean")
for p in sorted(clean_dir.glob("*.parquet")):
    df_tmp = pd.read_parquet(str(p))
    print(f"{p.name}: {len(df_tmp)} rows, columns: {list(df_tmp.columns)}")

news.parquet: 92 rows, columns: ['date', 'source', 'title', 'published_date', 'summary', 'sentiment_compound']
prices.parquet: 1 rows, columns: ['date', 'open', 'high', 'low', 'close', 'volume']


---
## ⚙️ Step 4: Spark — Feature Engineering
This is the KEY Spark step with **3 window functions**:
1. **7-day rolling average** of oil close price
2. **3-day rolling average** of Reddit/news sentiment
3. **Price delta** from previous day (lag function)

Also joins price + sentiment + news on date, and creates the `price_direction` label.

In [9]:
!python scripts/transform/spark_features.py

Features saved: 62 rows
SUCCESS: The process with PID 13244 (child process of PID 22912) has been terminated.
SUCCESS: The process with PID 22912 (child process of PID 27404) has been terminated.
SUCCESS: The process with PID 27404 (child process of PID 5712) has been terminated.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 1) / 1]

[Stage 0:===========================================================(1 + 0) / 1]

                                                                                
26/04/05 18:03:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/05 18:03:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.

[Stage 2:>                                                          (0 + 1) / 1]
26/04/05 18:03:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/05 18:03:37 WARN WindowExec: No Partition

In [11]:
# View the features — this is what the ML model will use
features_path = "data/processed/features/features.parquet"
if Path(features_path).exists():
    df_feat = pd.read_parquet(features_path)
    print(f"Features: {len(df_feat)} rows, {len(df_feat.columns)} columns")
    print(f"Columns: {list(df_feat.columns)}")
    print(f"\nSample (last 10 days):")
    print(df_feat.tail(10))

Features: 62 rows, 12 columns
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'rolling_avg_7d', 'price_delta', 'reddit_sentiment_avg', 'sentiment_rolling_3d', 'news_sentiment', 'price_direction']

Sample (last 10 days):
          date        open        high        low       close  volume  \
52  2026-03-20   95.000000   99.669998  93.419998   98.320000  461845   
53  2026-03-23  100.510002  101.669998  84.370003   88.129997  666790   
54  2026-03-24   88.779999   93.360001  86.339996   92.349998  420067   
55  2026-03-25   88.489998   91.730003  86.459999   90.320000  405170   
56  2026-03-26   91.379997   95.440002  89.510002   94.480003  350768   
57  2026-03-27   93.309998  101.239998  92.080002   99.639999  374920   
58  2026-03-30  102.599998  105.360001  99.430000  102.879997  374075   
59  2026-03-31  105.070000  106.860001  99.620003  101.379997  495395   
60  2026-04-01  101.720001  103.309998  96.500000  100.120003  434561   
61  2026-04-02   98.919998  113.970001

---
## 📊 Step 5: Spark — Aggregate Daily Summary
Creates a rolled-up daily summary for the dashboard.  
Outputs Parquet (full history) + CSV (last 30 days).

In [12]:
!python scripts/transform/spark_aggregate.py

Aggregated: 62 total days, 30 in CSV
SUCCESS: The process with PID 10752 (child process of PID 11796) has been terminated.
SUCCESS: The process with PID 11796 (child process of PID 11516) has been terminated.
SUCCESS: The process with PID 11516 (child process of PID 3436) has been terminated.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 1) / 1]

                                                                                

[Stage 1:>                                                          (0 + 1) / 1]

[Stage 1:===========================================================(1 + 0) / 1]

                                                                                


In [13]:
# View aggregated summary
agg_csv = "data/processed/aggregated/daily_summary_30d.csv"
if Path(agg_csv).exists():
    df_agg = pd.read_csv(agg_csv)
    print(f"Aggregated: {len(df_agg)} days")
    print(df_agg.tail(10))

Aggregated: 30 days
          date  avg_price  min_price  max_price  avg_reddit_sentiment  \
20  2026-03-20      98.32      93.42      99.67                   0.0   
21  2026-03-23      88.13      84.37     101.67                   0.0   
22  2026-03-24      92.35      86.34      93.36                   0.0   
23  2026-03-25      90.32      86.46      91.73                   0.0   
24  2026-03-26      94.48      89.51      95.44                   0.0   
25  2026-03-27      99.64      92.08     101.24                   0.0   
26  2026-03-30     102.88      99.43     105.36                   0.0   
27  2026-03-31     101.38      99.62     106.86                   0.0   
28  2026-04-01     100.12      96.50     103.31                   0.0   
29  2026-04-02     112.06      97.50     113.97                   0.0   

    avg_news_sentiment  rolling_avg_7d  avg_price_delta  price_direction  
20                 0.0           96.42             2.18                0  
21                 0.0    

---
## 🤖 Step 6: ML — Train Model
Trains a RandomForestClassifier on the features.  
Target: `price_direction` (1=Up, 0=Down).  
Uses `shuffle=False` in train/test split (time-series — no data leakage).

In [14]:
!python scripts/ml/train_model.py

Accuracy:  0.4615
Precision: 0.5000
Recall:    0.8571
Train size: 49, Test size: 13
Model saved to C:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\models\rf_model.pkl
Metrics saved to C:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\models\metrics.json


In [15]:
# View saved metrics
import json
metrics_path = "models/metrics.json"
if Path(metrics_path).exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("Model Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.2%}" if isinstance(v, float) else f"  {k}: {v}")

Model Metrics:
  accuracy: 46.15%
  precision: 50.00%
  recall: 85.71%


---
## 🔮 Step 7: ML — Predict Today
Loads the trained model and predicts today's oil price direction.

In [16]:
!python scripts/ml/predict.py

Prediction for 2026-04-02: Up (confidence: 69.48%)
Saved to C:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\data\processed\predictions\2026-04-02.json


c:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\scripts\ml\predict.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pred_date = str(latest.get("date", datetime.utcnow().strftime("%Y-%m-%d")))


In [17]:
# View today's prediction
pred_files = sorted(glob.glob("data/processed/predictions/*.json"))
if pred_files:
    with open(pred_files[-1]) as f:
        pred = json.load(f)
    print(f"📅 Date:       {pred['date']}")
    print(f"📈 Prediction: {pred['prediction']}")
    print(f"🎯 Confidence: {pred['confidence']:.1%}")
    print(f"\nFeatures used:")
    for k, v in pred['features'].items():
        print(f"  {k}: {v}")

📅 Date:       2026-04-02
📈 Prediction: Up
🎯 Confidence: 69.5%

Features used:
  price_delta: 11.94
  rolling_avg_7d: 100.1257
  sentiment_rolling_3d: 0.0
  news_sentiment: 0.0


---
## 🔮 Step 7b: War Exit Pressure Model
Forecasts likelihood of Trump leaving the Middle East conflict based on his base's opinion.  
Uses sentiment from r/conservative + r/politics and war news stance ratios.  
*This is a sentiment-based analytical tool, not a political forecast.*

In [ ]:
!python scripts/ml/war_exit_model.py

In [ ]:
# View war exit prediction
war_exit_files = sorted(glob.glob("data/processed/predictions/war_exit_*.json"))
if war_exit_files:
    with open(war_exit_files[-1]) as f:
        we = json.load(f)
    print(f"📅 Date:             {we['date']}")
    print(f"🎯 Exit Probability: {we['exit_probability']:.1%}")
    print(f"📊 Pressure Index:   {we['pressure_index']:.1%}")
    print(f"📈 Base Trend:       {we['base_sentiment_trend']}")
    print(f"⚖️  Favor Ratio:      {we['favor_ratio']:.1%}")
else:
    print("No war exit predictions yet.")

---
## 💾 Step 8: Load into DuckDB
Loads aggregated data + predictions into DuckDB with upsert logic.  
This is what the Streamlit dashboard reads from.

In [18]:
!python scripts/ml/load_duckdb.py

Loaded 30 rows into daily_summary
Loaded 1 rows into predictions
Loaded 1 rows into predictions
DuckDB updated at C:\Users\yaniv\OneDrive\שולחן העבודה\עבודה\oil-pulse-pipeline\data\oil_pulse.duckdb


In [ ]:
# Query DuckDB directly to verify all tables
import duckdb

db_path = "data/oil_pulse.duckdb"
if Path(db_path).exists():
    con = duckdb.connect(db_path, read_only=True)
    
    print("=== daily_summary (last 5 rows) ===")
    print(con.execute("SELECT * FROM daily_summary ORDER BY date DESC LIMIT 5").fetchdf())
    
    print("\n=== predictions ===")
    print(con.execute("SELECT * FROM predictions ORDER BY date DESC").fetchdf())
    
    # New tables
    for table in ["trump_statements", "war_news", "war_exit_index"]:
        try:
            df_t = con.execute(f"SELECT COUNT(*) as cnt FROM {table}").fetchdf()
            print(f"\n=== {table}: {df_t['cnt'].iloc[0]} rows ===")
            print(con.execute(f"SELECT * FROM {table} ORDER BY date DESC LIMIT 3").fetchdf())
        except Exception:
            print(f"\n=== {table}: not yet created ===")
    
    con.close()
else:
    print("DuckDB file not found yet.")

=== daily_summary (last 5 rows) ===
        date  avg_price  min_price  max_price  avg_reddit_sentiment  \
0 2026-04-02     112.06      97.50     113.97                   0.0   
1 2026-04-01     100.12      96.50     103.31                   0.0   
2 2026-03-31     101.38      99.62     106.86                   0.0   
3 2026-03-30     102.88      99.43     105.36                   0.0   
4 2026-03-27      99.64      92.08     101.24                   0.0   

   avg_news_sentiment  rolling_avg_7d  avg_price_delta  price_direction  
0                 0.0          100.13            11.94                0  
1                 0.0           97.31            -1.26                1  
2                 0.0           95.60            -1.50                0  
3                 0.0           95.16             3.24                0  
4                 0.0           94.20             5.16                1  

=== predictions ===
        date prediction  prediction_numeric  confidence
0 2026-04-02    

---
## ✅ Pipeline Complete!

**What we built:**
- 5 data sources: Yahoo Finance, BBC/CNBC/MarketWatch, Google News (Trump + War), Al Jazeera, Reddit (7 subs)
- Spark: 5 window functions, topic tagging, favor/against classification
- ML: Price direction model (RandomForest) + War Exit Pressure model (sentiment-based)
- DuckDB: 5 tables (daily_summary, predictions, trump_statements, war_news, war_exit_index)
- Dashboard: 7-section Streamlit app with gauges, sentiment grid, and news ticker

**Next steps:**
- Launch the dashboard: run `streamlit run dashboard/app.py` in a terminal
- Run tests: `pytest tests/ -v`
- Start Airflow: `astro dev init` then `astro dev start`